In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.9)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()
thread_id = 1
config = {"configurable": {"thread_id": "1"}}

builder = StateGraph(ChatState)

builder.add_node("chat_node", chat_node)

builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)

graph = builder.compile(checkpointer=checkpointer)
graph

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='what is my name?')]
}

graph.invoke(initial_state, config=config)